<a href="https://colab.research.google.com/github/JoshuaCongHu/CTRL-Freaks/blob/Jayda/phase2/src/phase2_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Who Waits Longer in Buffalo? Measuring Fairness in 311 Response Times Across Neighborhoods
**CSE 487 Project Phase 1: Data Cleaning & Processing**
**Team CTRL-Freaks:** Jayda Johnson, Meena Kasina, Josh Hu

**Research question:** Are 311 response times fair across Buffalo's council districts and neighborhoods from 2020 to 2026?

### Data Sources
This notebook loads live, externally hosted data directly from the City of Buffalo Open Data Portal, so no data files are included in the submission.
- [311 Service Requests (July 2008 to May 2024)](https://data.buffalony.gov/Quality-of-Life/311-Service-Requests-July-2008-May-2024-/whkc-e5vr/about_data)
- [311 Service Requests (May 2024 to Present)](https://data.buffalony.gov/Quality-of-Life/311-Service-Requests-May-2024-Present-/3tj7-3tdz/about_data)

### Cleaning & Processing Pipeline
The operations run in this order because some steps depend on earlier ones:

| # | Operation | Why it runs here |
|---|---|---|
| 1 | Data Integration | Combines both datasets so every later step runs on one table |
| 2 | Removing Irrelevant Data | Trims rows and columns before any heavier processing |
| 3 | Data Type Conversion | Dates and coordinates must be real types before they can be checked |
| 4 | String Cleaning | Cleans text so later grouping and duplicate checks match correctly |
| 5 | Fixing Swapped & Placeholder Coordinates | Creates new missing values that Operation 6 then handles |
| 6 | Handling Missing Data | Deals with all missing values, including those from Operation 5 |
| 7 | Removing Duplicates | Uses the original request types, so it runs before they are grouped |
| 8 | Category Standardization | Groups request types into broader categories last |

A final set of checks at the end confirms each operation did what it was supposed to.

In [4]:
import os
import numpy as np
import pandas as pd

## Loading the Data
Both datasets are read straight from the City of Buffalo Open Data Portal as CSV files. `low_memory=False` tells pandas to read each column in one pass so it doesn't guess mixed data types.

*Note: the older dataset is large (about 1.1 million rows), so this cell can take a minute or two to run.*

In [5]:
# URLs to the online data files
old_url = "https://data.buffalony.gov/api/views/whkc-e5vr/rows.csv?accessType=DOWNLOAD"
new_url = "https://data.buffalony.gov/api/views/3tj7-3tdz/rows.csv?accessType=DOWNLOAD"

# Load both datasets directly from the portal
oldDf = pd.read_csv(old_url, low_memory=False)
newDf = pd.read_csv(new_url, low_memory=False)

print("July 2008-May 2024: ",oldDf.shape)
print("May 2024-Present: ",newDf.shape)

July 2008-May 2024:  (1129274, 34)
May 2024-Present:  (206950, 28)


## Operation 1: Data Integration
**What:** Merge the two 311 datasets into one dataset covering January 2020 to present.

**Why:** Our time range is split across two files, and the old and new city systems store the same information under different column names.

**How:**
1. Filter the July 2008 to May 2024 data to requests opened on or after January 1, 2020
2. Rename the newer dataset's columns to match the older dataset (ex: `Created Date` → `Open Date`, `CaseNumber` → `Case Reference`)
3. Keep only the columns both datasets share, plus `Duplicate Request` (new data) and `Council District 2011` (old data), which later operations need
4. Add a `Source` column so each row can be traced back to its original file
5. Stack the two datasets with `pd.concat()`

In [6]:
# Step 1: Filter oldDf to January 2020 and later
# We parse "Open Date" into a temporary datetime Series just to filter.
# We do NOT overwrite the original column, since converting date
# columns is its own cleaning step (Data Type Conversion).
old_open_dates = pd.to_datetime(oldDf["Open Date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
oldDf = oldDf[old_open_dates >= "2020-01-01"].copy()
print("oldDf after filtering to 2020+:", oldDf.shape)

oldDf after filtering to 2020+: (362538, 34)


In [7]:
# Step 2: Rename newDf's columns to match oldDf's names
# Left side = newDf's name, right side = oldDf's name for the same info.
column_mapping = {
    "Created Date": "Open Date",
    "CaseNumber": "Case Reference",
    "Department": "Subject",
    "PropertyID": "Property ID",
    "Assessed Number": "Address Number",
    "AssessedStreet": "Address Line 1",
    "Zip": "Zipcode",
    "PoliceDistrict": "Police District",
    "PlanningNeighborhood": "Neighborhood",
}
newDf = newDf.rename(columns=column_mapping)

In [8]:
# Step 3: Keep only the columns both datasets share
# Columns that only exist in one file would be empty (NaN) for the
# other file's rows after stacking, so we keep the shared ones only.
shared_columns = [
    "Case Reference",
    "Open Date",
    "Closed Date",
    "Status",
    "Subject",
    "Type",
    "Address Number",
    "Address Line 1",
    "City",
    "State",
    "Zipcode",
    "Property ID",
    "Latitude",
    "Longitude",
    "Location",
    "Council District",
    "Police District",
    "Census Tract",
    "Census Block Group",
    "Census Block",
    "Neighborhood",
]
# The old data also has "Council District 2011", which Op 4 uses to fill in
# missing council districts, so we keep it too (new rows will be NaN).
oldDf = oldDf[shared_columns + ["Council District 2011"]]

# newDf also has a "Duplicate Request" flag that we need later for
# Duplicate Removal, so we keep it alongside the shared columns.
newDf = newDf[shared_columns + ["Duplicate Request"]]

In [9]:
# Step 4: Tag each row with the dataset it came from
# This makes it easy to trace any issue back to its original file.
oldDf = oldDf.assign(Source="July 2008-May 2024 dataset")
newDf = newDf.assign(Source="May 2024-Present dataset")

In [10]:
# Step 5: Stack the two datasets into one
# pd.concat places newDf's rows underneath oldDf's rows.
# ignore_index=True renumbers the rows from 0 to the end.
# oldDf rows will show NaN for "Duplicate Request" since the old
# dataset did not track duplicates.
merged_df = pd.concat([oldDf, newDf], ignore_index=True)
print("Merged Dataset:", merged_df.shape)

Merged Dataset: (569488, 24)


## Operation 2: Removing Irrelevant Data
**What:** Remove rows and columns that don't help answer our research question.

**Why:** Test entries aren't real requests, and extra columns add clutter without adding value.

**How:**
- **Rows:** Drop the 4 test entries where the department (`Subject`) is `"Test"`. (The 2020+ date filter already happened in Operation 1.)
- **Columns:** Drop columns that repeat other information or aren't needed:
  - `Location`: same information as `Latitude` and `Longitude` in a different format
  - `City`, `State`: every row is Buffalo, NY
  - `Census Block Group`, `Census Block`: more specific than our analysis needs
  - `Property ID`: parcel IDs aren't needed for our question

**Kept on purpose:** `Census Tract` (so we can later compare requests per resident using Census population data), `Council District 2011` (used in Operation 6), and `Duplicate Request` (used in Operation 7).

In [11]:
# Start cleaning from the merged dataset created in Operation 1
df = merged_df.copy()
print("Shape before:", df.shape)

# Double-check that the 2020+ filter from Operation 1 worked
openDates = pd.to_datetime(df["Open Date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
print("Earliest date:", openDates.min())
print("Latest date:", openDates.max())

Shape before: (569488, 24)
Earliest date: 2020-01-01 10:28:00
Latest date: 2026-10-18 05:30:00


In [12]:
# Remove test entries, which aren't real requests
testRows = df["Subject"] == "Test"
print("Test rows removed:", testRows.sum())
df = df[~testRows]

# Drop columns that repeat other information or aren't needed
dropCols = ["Location", "Census Block Group", "Census Block", "City", "State", "Property ID"]
df = df.drop(columns=dropCols)

print("Shape after:", df.shape)
print(list(df.columns))

Test rows removed: 4
Shape after: (569484, 18)
['Case Reference', 'Open Date', 'Closed Date', 'Status', 'Subject', 'Type', 'Address Number', 'Address Line 1', 'Zipcode', 'Latitude', 'Longitude', 'Council District', 'Police District', 'Census Tract', 'Neighborhood', 'Council District 2011', 'Source', 'Duplicate Request']


## Operation 3: Data Type Conversion
**What:** Convert columns to their correct data types.

**Why:** Dates stored as text can't be used to calculate response times, which is the core of our project.

**How:**
- `Open Date`, `Closed Date` → datetime (both files use the format `09/03/2026 02:15:00 PM`)
- `Latitude`, `Longitude` → numeric
- `Duplicate Request` → True/False (old rows are blank because the old system didn't have this flag, so they become False)
- `Census Tract` → one consistent 6-digit code. The new data already uses codes like `002400`, but the old data writes the same tract as a decimal like `24` or `16.02`. Multiplying old values by 100 and padding with zeros gives the matching code (`24` → `002400`, `16.02` → `001602`). Anything that still isn't 6 digits (`UNKNOWN` or broken codes) becomes missing.

`errors="coerce"` turns anything that can't be converted into a missing value instead of crashing, and we count how many that happened to.

In [13]:
print("Data types before conversion:")
print(df.dtypes)

Data types before conversion:
Case Reference            object
Open Date                 object
Closed Date               object
Status                    object
Subject                   object
Type                      object
Address Number            object
Address Line 1            object
Zipcode                   object
Latitude                 float64
Longitude                float64
Council District          object
Police District           object
Census Tract              object
Neighborhood              object
Council District 2011     object
Source                    object
Duplicate Request         object
dtype: object


In [14]:
dateFormat = "%m/%d/%Y %I:%M:%S %p"
df["Open Date"] = pd.to_datetime(df["Open Date"], format=dateFormat, errors="coerce")
df["Closed Date"] = pd.to_datetime(df["Closed Date"], format=dateFormat, errors="coerce")
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

# Old rows are blank here (no duplicate flag in the old system), so they become False
df["Duplicate Request"] = df["Duplicate Request"].astype(str).str.lower() == "true"

print("Data types after conversion:")
print(df.dtypes)

# Open Date should have 0 failures. A missing Closed Date means the case is still open, not a failed conversion
print("Open Date that failed to parse:", df["Open Date"].isna().sum())
print("Closed Date empty (open cases):", df["Closed Date"].isna().sum())

Data types after conversion:
Case Reference                   object
Open Date                datetime64[ns]
Closed Date              datetime64[ns]
Status                           object
Subject                          object
Type                             object
Address Number                   object
Address Line 1                   object
Zipcode                          object
Latitude                        float64
Longitude                       float64
Council District                 object
Police District                  object
Census Tract                     object
Neighborhood                     object
Council District 2011            object
Source                           object
Duplicate Request                  bool
dtype: object
Open Date that failed to parse: 0
Closed Date empty (open cases): 17329


In [15]:
# Census tract is written two ways: old = decimal like 24 or 16.02, new = 6-digit text like "002400"
fromOld = df["Source"] == "July 2008-May 2024 dataset"
tract = df["Census Tract"].astype(str).str.strip()

# Old codes: multiply by 100 and pad with zeros to match (24 -> 002400, 16.02 -> 001602)
oldNum = pd.to_numeric(tract[fromOld], errors="coerce") * 100
tract[fromOld] = oldNum.round().apply(lambda x: f"{int(x):06d}" if pd.notna(x) else np.nan)

# Anything that still isn't 6 digits is invalid (UNKNOWN or broken 4-5 digit codes in the new data)
badTract = ~tract.str.fullmatch(r"\d{6}", na=False)
print("Invalid tract codes set to missing:", badTract.sum())
tract[badTract] = np.nan
df["Census Tract"] = tract

print("Distinct tracts:", df["Census Tract"].nunique())
print(df["Census Tract"].value_counts().head())

Invalid tract codes set to missing: 23826
Distinct tracts: 126
Census Tract
000200    15903
002400    13486
004300    11552
003700    11148
004500    11012
Name: count, dtype: int64


## Operation 4: String Cleaning
**What:** Clean up inconsistent text formatting and spelling in the `Type` and `Subject` (department) columns.

**Why:** The same department appears under different names (ex: `Parking` and `Dept of Parking`), which would split its requests into separate groups.

**How:**
- Remove the `(Req_Serv)` suffix from request types, since every row is already a service request
- Combine department name variants into one consistent name
- Label missing departments as `Unknown`

In [16]:
# Remove the "(Req_Serv)" suffix from request types, since every row is already a service request
df['Type'] = df['Type'].str.replace(' (Req_Serv)', '', regex=False)

In [17]:
# Combine department name variants into one consistent name
updated_request_map = {"Mayor's Office": "Office of the Mayor",
                       "Parking" : "Parking Department",
                       "Dept of Parking" : "Parking Department",
                       "Dept of Public Works": "Public Works, Parks, & Streets Department",
                       "Public Works, Parks & Streets" : "Public Works, Parks, & Streets Department",
                       "Community Services & Rec. Program.": "Community Service & Recreational Programming",
                       "Community Services & Recreational Programming": "Community Service & Recreational Programming",
                       "DPIS" : "Permit & Inspection Services",
                       "Dept of Law" : "Law Department"
}
df['Subject'] = df['Subject'].replace(updated_request_map)
# Label missing departments as Unknown
df['Subject'] = df['Subject'].fillna('Unknown')

## Operation 5: Fixing Swapped & Placeholder Coordinates
**What:** Correct invalid values in `Latitude` and `Longitude`.

**Why:** Bad coordinates would place requests in the wrong location on maps and in any location-based analysis.

**How:**
1. **Swapped coordinates (new data):** About 10,000 rows have latitude around -78.8 and longitude around 42.9. Buffalo's latitude is positive (about 42.9), so any row with a negative latitude was flipped, and we swap it back.
2. **Placeholder coordinates (old data):** About 85% of old rows (nearly all of 2020 to 2023) are stored at exactly (43.0, -79.0). This isn't a real location; it's a default the old system filled in. Leaving it would stack 300,000+ requests on one fake point, so we set these to missing. These rows still have a council district and neighborhood, so they remain usable.
3. **Boundary check:** Any remaining coordinates outside Buffalo (latitude 42.82 to 42.97, longitude -78.92 to -78.79, based on the range of real coordinates) are set to missing.

*This runs before Operation 6 because it creates new missing values that Operation 6 then handles.*

In [18]:
# Buffalo's latitude is positive (about 42.9), so a negative latitude means lat and lon were swapped
swapped = df["Latitude"] < 0
print("Rows with swapped coordinates:", swapped.sum())
print(df.loc[swapped, "Source"].value_counts())

tempLat = df.loc[swapped, "Latitude"]
df.loc[swapped, "Latitude"] = df.loc[swapped, "Longitude"]
df.loc[swapped, "Longitude"] = tempLat

print("Rows with negative latitude after fix:", (df["Latitude"] < 0).sum())

Rows with swapped coordinates: 10126
Source
May 2024-Present dataset    10126
Name: count, dtype: int64
Rows with negative latitude after fix: 0


In [19]:
# (43.0, -79.0) appears 300,000+ times. It's a default placeholder, not a real location
print(df["Latitude"].value_counts().head(3))

placeholder = (df["Latitude"] == 43) & (df["Longitude"] == -79)
print("Placeholder coordinate rows:", placeholder.sum())
print(df.loc[placeholder, "Open Date"].dt.year.value_counts().sort_index())

df.loc[placeholder, "Latitude"] = np.nan
df.loc[placeholder, "Longitude"] = np.nan

Latitude
43.000000    309526
42.913366       404
42.912594       307
Name: count, dtype: int64
Placeholder coordinate rows: 309526
Open Date
2020    67441
2021    72826
2022    97446
2023    71813
Name: count, dtype: int64


In [20]:
# Any coordinates outside Buffalo's boundaries are treated as invalid
inBuffalo = df["Latitude"].between(42.82, 42.97) & df["Longitude"].between(-78.92, -78.79)
outside = df["Latitude"].notna() & ~inBuffalo
print("Coordinates outside Buffalo:", outside.sum())

df.loc[outside, "Latitude"] = np.nan
df.loc[outside, "Longitude"] = np.nan

print("Rows with valid coordinates:", df["Latitude"].notna().sum(), "/", len(df))

Coordinates outside Buffalo: 0
Rows with valid coordinates: 239953 / 569484


## Operation 6: Handling Missing Data
**What:** Find all missing values and decide how to handle each column.

**Why:** Both city systems write the text `"UNKNOWN"` (and sometimes `"UnAssigned"`) instead of leaving cells blank, so pandas doesn't recognize them as missing and `isna()` undercounts them.

**How:**
1. Convert `"UNKNOWN"`, `"UnAssigned"`, and empty text to true missing values, then compare missing counts before and after
2. **`Council District`:** The old system left this blank for most of 2020 to 2021. `Council District 2011` is filled for most of those rows, so we use it to fill the gaps and then drop it. *Limitation: 2011 district lines don't perfectly match current ones, so a small share of filled-in districts may be off.*
3. **Rows still missing `Council District`:** Dropped, since comparing districts is the point of our project
4. **Missing `Closed Date`:** These are still-open cases, not bad data. We keep them and flag them with a new `isOpen` column so response time analysis can skip them
5. **`Latitude`, `Longitude`, `Zipcode`, `Neighborhood`, `Police District`, address columns:** Left as missing, since not every analysis needs them and dropping those rows would throw away useful data

In [21]:
# "UNKNOWN" is a missing value that isna() can't detect, so convert it to NaN
missingBefore = df.isna().sum()
unknownBefore = (df == "UNKNOWN").sum() + (df == "UnAssigned").sum()

df = df.replace(["UNKNOWN", "UnAssigned", ""], np.nan)

missingTable = pd.DataFrame({
    "Missing before": missingBefore,
    "UNKNOWN text": unknownBefore,
    "Missing after": df.isna().sum(),
    "% missing": (df.isna().sum() / len(df) * 100).round(2)})
missingTable

,Missing before,UNKNOWN text,Missing after,% missing
Case Reference,0,0,0,0.00
Open Date,0,0,0,0.00
Closed Date,17329,0,17329,3.04
Status,0,0,0,0.00
Subject,0,0,0,0.00
Type,0,0,0,0.00
Address Number,22465,0,22465,3.94
Address Line 1,3828,0,3828,0.67
Zipcode,10769,19753,30522,5.36
Latitude,329531,0,329531,57.86


In [22]:
# Fill missing council districts using the 2011 district column, then drop it
print("Council District missing before:", df["Council District"].isna().sum())
df["Council District"] = df["Council District"].fillna(df["Council District 2011"])
print("Council District missing after fill:", df["Council District"].isna().sum())

df = df.drop(columns=["Council District 2011"])

Council District missing before: 160171
Council District missing after fill: 22458


In [23]:
# Rows with no council district can't be compared, so they are dropped
noDistrict = df["Council District"].isna()
print("Rows with no district dropped:", noDistrict.sum())
df = df[~noDistrict]

# Keep open cases, but flag them so response time analysis can skip them
df["isOpen"] = df["Closed Date"].isna()
print("Open cases kept:", df["isOpen"].sum())
print("Rows remaining:", len(df))

Rows with no district dropped: 22458
Open cases kept: 14582
Rows remaining: 547026


## Operation 7: Removing Duplicates
**What:** Remove repeat reports of the same issue.

**Why:** Duplicates inflate request counts for some areas, which would skew comparisons between districts.

**How:**
1. Check for exact duplicate rows and repeated case IDs (both are 0, and the two datasets' date ranges don't overlap)
2. Remove requests the new system flagged itself as duplicates (`Duplicate Request = True`)
3. Remove near-duplicates: same request type, same address, same day, keeping the earliest report. Rows with no street address are skipped, since we can't confirm they're the same location.

*This runs before Operation 8 so near-duplicates are matched on the original, specific request types.*

In [24]:
# Check for exact duplicate rows and repeated case IDs
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate case IDs:", df["Case Reference"].duplicated().sum())

Exact duplicate rows: 0
Duplicate case IDs: 0


In [25]:
# Remove requests the city's new system flagged as duplicates
flagged = df["Duplicate Request"] == True
print("Flagged duplicates removed:", flagged.sum())
df = df[~flagged]
df = df.drop(columns=["Duplicate Request"])

Flagged duplicates removed: 4876


In [26]:
# Same type, same address, same day = the same issue reported more than once
# Sort by time first so keep="first" keeps the earliest report
df = df.sort_values("Open Date")
df["createdDay"] = df["Open Date"].dt.date

nearDupes = df.duplicated(subset=["Type", "Address Number", "Address Line 1", "createdDay"], keep="first")
# Skip rows with no street, since we can't confirm they're the same location
nearDupes = nearDupes & df["Address Line 1"].notna()

print("Near-duplicates removed:", nearDupes.sum())
print(df.loc[nearDupes, "Source"].value_counts())

df = df[~nearDupes]
df = df.drop(columns=["createdDay"])
df = df.reset_index(drop=True)
print("Rows remaining:", len(df))

Near-duplicates removed: 13838
Source
July 2008-May 2024 dataset    10861
May 2024-Present dataset       2977
Name: count, dtype: int64
Rows remaining: 528312


## Operation 8: Category Standardization
**What:** Group 300+ specific request types into broader categories.

**Why:** Many request types describe the same issue under different names (ex: `Pot Hole`, `Pothole Issue`, and `Other Hole in Road`), which makes comparisons by type unreliable.

**How:** Each request type is mapped into one of 24 categories (ex: `Roads & Potholes`, `Snow Management`). Anything that doesn't match a category is labeled `Other`.

In [27]:
print("Unique request types before grouping:", df['Type'].nunique())

# Waste categories
totes_bins = ['Totes Audit', 'Totes Deliver', 'Tote Pickup', 'Totes Pickup',
              'Totes Replace', 'Totes Combo', 'Totes Abandon Pickup',
              'Totes Report of Removed Broken Tote', 'Special Even Totes',
              'Order Additional or Change Current Totes', 'Replacement Garbage Totes Request',
              'Recycling Tote Replace', 'Recycyling Tote Deliver', 'Recycling Tote Pickup',
              'Recycling Tote Abandon Pickup', 'Recycling Tote Combo', 'Recycling Tote Combo (Exchange Size)','Impact Bill Inquiry or Concern',
              'Special Event Totes']
regular_bulk_collection = [
    "Garbage Missed Pick Up", "Recycling Missed Pick Up", "Missed Pickup 2_Piece Large Trash",
    "Missed Pickup; 2 Piece Large Trash", "Missed Pickup; 3 Piece Large Trash",
    "Bulk Trash", "Bulk Trash Issue", "Scrap_It Curbside",
    "Senior or Disabled Citizen, TV Pickup Request",
    "Senior or Disabled Citizen, TV/Electronics Pickup Request", "Electronic Waste",
    "TV and Electronic Waste at Curb", "Fridge", "Fridge Removal", "Christmas Tree",
    "Christmas Tree Removal", "Leaves / Lawn Debris", "High Grass/Weeds",
    "Shopping Carts (Impact Team)", 'Trash Ordinance Violation','Rain Barrels'
]
illegal_dumping_debris = [
    "Excess Trash", "Illegal Dumping Curb", "Illegal Dumping PrivateProperty",
    "Illegal Dumping", "OSP Illegal Dumping", "QRT Illegal Dumping on Viaduct",
    "Illegal Dumping Private Property / Vacant Lot", "Great American Clean-Up",
    "Neighborhood CleanUp", "Neighborhood CleanUp Request",
    "Weather Emergency Trash Pick Up - 9/3/2026 Event", 'Illegal Dumping Street'
]

Unique request types before grouping: 308


In [28]:
# Streets, sidewalks, and roads
roads_and_potholes = [
    "Pot Hole", "Other Hole in Road", "Pothole Issue", "Paving", "Street Paving Request",
    "Cave In", "PW Ongoing Construction", "PW Ongoing Construction Issue",
    "PW Engineering", "Obscene PW Engineering", 'Engineering Operational', 'Damage from Street Worker',
    'CityHall_CityCourt Maintenance'
]
sidewalks_and_curbs = [
    "Sidewalks", "New Sidewalk/Ramp", "Sidewalk(s) Repair Request", "Sidewalk(s) Issue",
    "Curbs", "Curbs Replacement Request", "Curb - Metal Protruding",
    "Curb - Metal Protruding Issue", "Bike Rack", "Bike Rack Issue", "Bike Rack Request"
]
lighting_and_signals = [
    "Streetlights", "Streetlight(s) Issues", "Damaged Street Light Pole",
    "Damaged or Missing Street Light Pole", "Signal Out or Flashing", "Signal Issue DoT",
    "Signal Other Issue", "Signal Issue", "Signal Timing Issue City",
    "Signal Timing Issue in City of Buffalo"
]
traffic_and_lines = [
    "Pavement Marking Lines", "Pavement Marking Issue", "Buffalo Traffic Violations",
    "PVB School-Zone Camera Citations", "PW Traffic", "Obscene PW Traffic",
    "PW Neighborhood Traffic Calming", "Neighborhood Traffic Calming", "Speeding Issue(s)",
    "Bridge Issue", "Report Bridge Flooding", "Weather Event Damage to ROW Infrastructure",
    "Report Snow on Bridge or Viaduct", "Sweeper", "Sweeper Request", "Seweeper Issue", "ROW Bike Lane ; on-road or cycle tracks Issue"

]
obstructions_and_hazards = [
    "Basketball Hoop in RoW", "Basketball Hoop in Street", "Right of Way Issue",
    "Sign Hazards", "Sign Maintenance", "Sign Issue", "Down Wire(s)"]

In [29]:
# Forests and parks
tree_maintenance = [ "Tree Removal",
    "Tree Removal (Req_SERV)", "Tree Trimming Request", "Fallen Tree Blocking RoW",
    "Tree Other", "Fallen Tree Inspection", "Tree Removal Quality Issue (Req_serv)",
    "Tree Removal Challenge", "Stump Removal", "Tree Planting Request",
    "Tree Removal per Inspector", "Tree Pruning Request", "Tree Pruning Private Property",
    "Tree Planting Quality Issue", "Stump Removal Quality Issue",
    "Tree Trimming Quality Issue", "City Park Tree Issue",
    "Fallen Tree, split tree, hanging limb",
    "Fallen Tree Inspection on Private Property"
]
parks_and_recreation = [
    "City Parks", "City Parks Issue", "Parks City", "Olmsted Parks", "Olmsted Parks Issue",
    "Parks Olmsted", "RalphWilson Park Construction", "Ralph Wilson Park Issue",
    "Pools and Splashpads", "Pools and Splashpads Issue", "Recreation Center",
    "Recreation Center Issue", "Youth Bureau Issue", "Buffalo Place Park Issue",
    "Naval Park Issue", "Erie Basin Marina Issue",
    "Shoreline Trail, North Buffalo Rails to Trails, Jesse Kregal Pathway Issue",
    "Circles Medians Triangles Issues (Contractor)", "Obscene Parks City",
    "Obscene Parks Olmsted", "Parks Buildings Maintenance",
    "Tifft Nature Preserve Issue  (Buffalo Museum of Science)",
    "Pat Sole Park (Peace Bridge Authority)", "Ice Rink Issue"]

In [30]:
water_services = [
    "Water Issue", "Water_Billing_Meter", "Buffalo Water Service Request",
    "Buffalo Water Service Issue", "Buffalo Water Billing Request",
    "Buffalo Water Testing Request", "Water Tested"
]
drainage_and_sewers = [
    "Sewer", "Sewer Issue", "Report Flooding", "Street Flooding",
    "Basement Flooding", "Basement Flooding Issue", "PW Missing Manhole Cover"
]


In [31]:
# Housing and building code violations
violations_and_inspections = [
    "Housing Violations", "Open311 Housing", "Rental Registration",
    "Healthy Homes / PRI Inspection", "Housing or Building Violations(Exterior)",
    "Housing or Building Violations(Interior)", "Electrical Issue",
    "Electrical Violation (Interior)", "Electrical Violation (Exterior)",
    "Heating or A/C Issues(Exterior)", "Heating or A/C Issues(Interior)",
    "Property Issue(s)  (Quality of Life Issue)", "Short Term Rental Issue",
    "Zoning Board of Appeals Concern", 'Lead Paint Inspection', 'Building Maintenance', 'License Concern','Rodents',
    'Rodent Control Issue'
]
vacant_and_abandoned_properties = [
    "PW Vacant Lot", "Vacant Lot Cutting", "Inrem Real Estate", "Real Estate Issue",
    "Boarding request", "Board Up request", "Abandoned Property",
    "BPD Request Board Up/Cleanup Task, For Impact Team"
]
graffiti_and_obscenity = [
    "Open311 Graffiti", "Graffiti issue", "Graffiti Private Property",
    "Obscene City Property", "Obscene Parks City", "Obscene Private Property",
    "Obscene Other", "Graffiti, City Property or Right of Way (Impact Team)"
]

In [32]:
# Parking & vehicles
parking_enforcement_and_rules = [
    "Parking Issues", "General Parking Issue(s)",
    "Illegal Parking (Quality of Life Issue)", "Unlicensed Street Vendor",
    "Parking Pay Station Issue(s)", "Parking Mobile App Issue(s)",
    "Parking Single Meter Issue(s)"]
meters_and_apps = [
    "User Fee", "Public Works User Fee Issue", "PVB Mobile App Issue",
    "PVB Mobile App Ticket", "PVB Single Meter", "PVB Pay Station"
]
abandoned_vehicles = [
    "Abandoned Vehicles", "Abandoned Vehicle Inspection",
    "Privately Owned Property, Abandoned Vehicle", "Unregistered Vehicle",
    "Abandoned Vehicles (Parking Dept)",
    "Vehicle Immobilized in Street (Parking Dept)"]

In [33]:
# Public safety, fire, and animals
animal_and_wildlife_control = [
    "Animals", "Dead Animal Removal", "Dead Animal Removal Request",
    "Dog License Issue", "Dog Animal Control Issue", "Wildlife Control Issue",
    "Bees and Hornets Animal Control Issue", "Personal Wildlife Trap Report",
    "Pest", "Pest_Private Trap", "Chicken Coop", "Chicken Issue",
    "Dog Licensing Issue", "Livestock Issue"
]
nuisance_and_quality_of_life = [
    "Excessive Noise", "Noise Nuisance (Loud Music)", "Loitering", "Panhandling",
    "Homeless Concern (Quality of Life Issue)", "Juvenile / Youth Concern",
    "Quality of Life Issue", "Quality of Life Issue (Other)", "Good Neighbor",
    "Sneakers Hanging", "ATV and or Dirt Bike", "Unlicensed Business"
]
fire_safety_and_police = [
    "Fire", "BFD Fire Prevention", "Buffalo Fire Prevention Request",
    "General Fire Issue", "Police Issue", "Police Issue (Other)",
    "Public Safety Support - (BFD)"
]

In [34]:
# Winter/weather
snow_management = [
    "Street Snow Plowing Issue", "Street Snow Plowing", "Street Snow Plowing Request",
    "Street Salting", "Street Salting Issue", "Street Salting Request",
    "QRT Snow Removal", "Snow Removal", "Snow Removal Inspection",
    "Sidewalk Snow Removal Inspection", "Sidewalk Snow/Ice Removal Inspection",
    "Snow Removal Bus Stop/Shelter", "Damage from Snow Removal",
    "Vehicle Blocking Snow Plow", "Snow Removal Concern", "Snow-Bank Removal",
    "BFD Snow on Hydrant", "Buffalo Fire Department Snow on Hydrant"]

In [35]:
# Internal or administrative issues
administrative_and_records = [
    "FOIL Records EDPIS", "FOIL Records Citizens Services",
    "FOIL Records Board of Ethics", "City Clerk Issue", "General City Clerk Issue",
    "Taxation Issue", "Assessment Issue", "2020 Reassessment",
    "Compensation & Benefits", "New Americans Request"
]

internal_and_operations = [
    "BMHA Issue", "Pick and Pay", "Open311 APP Issues",
    "Report Technical Issue for Buff311 App", "Telecommunications",
    "General Impact Team Request", "Johnnie B Wiley Issue (BPS)", "Schools",
    "Save Our Streets Program", "Other Adjudication Issue",
    "Outer Harbor Parkway (Empire State Development Corp)", "CBO",
    "Fair Housing Issue", "FairHsg Other", "Security Deposit",
    "Union Ship Canal Issue (BUDC)", "DPIS Department Operational Complaint",
    "Obscene Schools", "DPIS Personnel Complaint"
]

In [36]:
# Map each category name to its list of request types
mapping = {"Totes & Bins" : totes_bins,
           "Regular & Bulk Collection":regular_bulk_collection,
           "Illegal Dumping & Debris" : illegal_dumping_debris,
           "Roads & Potholes" : roads_and_potholes,
           "Sidewalks & Curbs": sidewalks_and_curbs,
           "Lighting & Signals" : lighting_and_signals,
           "Traffic & Lines": traffic_and_lines,
           "Obstruction & Hazards": obstructions_and_hazards,
           "Tree Maintenance": tree_maintenance,
           "Parks & Recreation":parks_and_recreation,
           "Water Services": water_services,
           "Drainage & Sewers": drainage_and_sewers,
           "Violations & Inspections": violations_and_inspections,
           "Vacant & Abandoned Properties": vacant_and_abandoned_properties,
           "Graffiti & Obscenity": graffiti_and_obscenity,
           "Parking Enforcement & Rules": parking_enforcement_and_rules,
           "Meters & Apps": meters_and_apps,
           "Abandoned Vehicles": abandoned_vehicles,
           "Animal & Wildlife Control" : animal_and_wildlife_control,
           "Nuisance & Quality of Life": nuisance_and_quality_of_life,
           "Fire Safety & Police" : fire_safety_and_police,
           "Snow Management": snow_management,
           "Administrative & Records": administrative_and_records,
           "Internal & Operations" : internal_and_operations
           }

In [37]:
# Assign each request type to its category, or "Other" if it doesn't match any list
def assign_grouping(type_data):
  for group,lst in mapping.items():
    if type_data in lst:
      return group
  return "Other"
df['Type'] = df['Type'].apply(assign_grouping)

In [38]:
print("Unique request types after grouping:", df['Type'].nunique())
df['Type'].unique()

Unique request types after grouping: 25


array(['Parking Enforcement & Rules', 'Violations & Inspections',
       'Nuisance & Quality of Life', 'Illegal Dumping & Debris',
       'Roads & Potholes', 'Animal & Wildlife Control',
       'Vacant & Abandoned Properties', 'Regular & Bulk Collection',
       'Totes & Bins', 'Internal & Operations', 'Lighting & Signals',
       'Abandoned Vehicles', 'Obstruction & Hazards', 'Other',
       'Tree Maintenance', 'Water Services', 'Fire Safety & Police',
       'Meters & Apps', 'Traffic & Lines', 'Administrative & Records',
       'Parks & Recreation', 'Snow Management', 'Drainage & Sewers',
       'Sidewalks & Curbs', 'Graffiti & Obscenity'], dtype=object)

## Final Checks
These checks confirm each operation did what it was supposed to. If any check fails, the notebook stops with an error.

In [39]:
# Each check confirms one operation worked as intended
assert (df["Latitude"] < 0).sum() == 0
assert df["Latitude"].dropna().between(42.82, 42.97).all()
assert (df == "UNKNOWN").sum().sum() == 0
assert df["Council District"].isna().sum() == 0
assert str(df["Open Date"].dtype).startswith("datetime64")
assert str(df["Closed Date"].dtype).startswith("datetime64")
assert df["Case Reference"].duplicated().sum() == 0
assert (df["Census Tract"].dropna().str.len() == 6).all()

print("All checks passed!")
print("Final shape:", df.shape)
print(df["Source"].value_counts())

All checks passed!
Final shape: (528312, 17)
Source
July 2008-May 2024 dataset    331462
May 2024-Present dataset      196850
Name: count, dtype: int64


# Phase 2, Split C: What causes some requests to be answered slowly?
Everything above is our Phase 1 notebook as submitted (minus the EDA and the optional save step, to keep runtime down). Phase 2 starts here.

# Shared Phase 2 Features
These features are the same in all three Phase 2 notebooks, so everyone's models run on identical data. **Don't change this cell in your own notebook.** If a change is needed, tell the team so we update all three.

| Feature | What it is |
|---|---|
| `resolutionDays` | Days from Open Date to Closed Date (missing for still-open cases) |
| `openYear`, `openMonth` | Year and month the request was opened |
| `season` | Winter, Spring, Summer, or Fall |
| `afterSwitch` | True if opened on or after May 11, 2024, when Buffalo switched 311 systems |
| `isSlow` | True if a closed request took longer than the median for its request category (missing for open cases) |

In [40]:
# Resolution time in days (still-open cases stay missing)
df["resolutionDays"] = (df["Closed Date"] - df["Open Date"]).dt.total_seconds() / 86400

# Time features
df["openYear"] = df["Open Date"].dt.year
df["openMonth"] = df["Open Date"].dt.month
seasons = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
           6: "Summer", 7: "Summer", 8: "Summer", 9: "Fall", 10: "Fall", 11: "Fall"}
df["season"] = df["openMonth"].map(seasons)

# Before vs. after Buffalo's 311 system switch in May 2024
df["afterSwitch"] = df["Open Date"] >= "2024-05-11"

# Slow = took longer than the median for its request category (closed cases only),
# so a request is only called slow compared to similar requests
typeMedian = df.groupby("Type")["resolutionDays"].transform("median")
df["isSlow"] = (df["resolutionDays"] > typeMedian).where(df["resolutionDays"].notna())

print("Closed cases with a resolution time:", df["resolutionDays"].notna().sum())
print("Median resolution time (days):", round(df["resolutionDays"].median(), 2))
print("Share of closed requests labeled slow:", round(df["isSlow"].mean(), 3))
df[["resolutionDays", "openYear", "openMonth", "season", "afterSwitch", "isSlow"]].head()

Closed cases with a resolution time: 514373
Median resolution time (days): 3.99
Share of closed requests labeled slow: 0.5


,resolutionDays,openYear,openMonth,season,afterSwitch,isSlow
0,1.035417,2020,1,Winter,False,True
1,1.013194,2020,1,Winter,False,True
2,2.063889,2020,1,Winter,False,False
3,0.901389,2020,1,Winter,False,False
4,1.093750,2020,1,Winter,False,True


# Algorithm 1: Random Forest
**Question it answers:** What matters most for slow requests: location, request type, or department? (Q2)

*Outside-class algorithm: cite Breiman (2001) or the Python Data Science Handbook, Ch. 5 in the report.*

In [51]:
# Step 1: Prep the data
import pandas as pd
from sklearn.model_selection import train_test_split

# Only closed requests have a resolution time, so they can be labeled slow or fast
rf_df = df[df["isOpen"] == False]

# 1 = slow, 0 = fast
y = rf_df["isSlow"].astype(int)

# Possible causes of slow service: location, request type, department, time of year, and the 2024 system switch
X = pd.get_dummies(rf_df[["Council District", "Type", "Subject", "season", "afterSwitch"]])

# Hold out 20% of requests to use for testing data
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

print("Features:", X.shape[1], "| Training rows:", len(Xtrain), "| Test rows:", len(Xtest))
print("Share of requests that are slow:", round(y.mean(), 3))

Features: 58 | Training rows: 411498 | Test rows: 102875
Share of requests that are slow: 0.5


In [52]:
# Step 2: Train a baseline Random Forest and grade it
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

# 100 trees vote on each request. max_depth=15 stops any tree from memorizing small details
model = RandomForestClassifier(n_estimators=100, max_depth=15, n_jobs=-1, random_state=0)
model.fit(Xtrain, ytrain)

# predict the test requests the model has never seen
ypred = model.predict(Xtest)

print("Accuracy:", round(metrics.accuracy_score(ytest, ypred), 3))
print(metrics.classification_report(ytest, ypred, target_names=["Fast", "Slow"]))

Accuracy: 0.609
              precision    recall  f1-score   support

        Fast       0.60      0.64      0.62     51448
        Slow       0.62      0.57      0.59     51427

    accuracy                           0.61    102875
   macro avg       0.61      0.61      0.61    102875
weighted avg       0.61      0.61      0.61    102875



In [53]:
# Step 3: Tune tree depth (improving the 61% accuracy since that's slightly more than guessing)
# If training accuracy is much higher than test accuracy, the trees are memorizing (overfitting)
for depth in [5, 10, 15, 20, None]:
    m = RandomForestClassifier(n_estimators=100, max_depth=depth, n_jobs=-1, random_state=0)
    m.fit(Xtrain, ytrain)
    print(f"max_depth={depth}: train={m.score(Xtrain, ytrain):.3f}, test={m.score(Xtest, ytest):.3f}")

max_depth=5: train=0.565, test=0.565
max_depth=10: train=0.601, test=0.599
max_depth=15: train=0.613, test=0.609
max_depth=20: train=0.617, test=0.611
max_depth=None: train=0.619, test=0.611


In [54]:
# Step 4: Add year and month as features, then train the final model
X = pd.get_dummies(rf_df[["Council District", "Type", "Subject", "season", "afterSwitch",
                          "openYear", "openMonth"]].astype({"openYear": str, "openMonth": str}))
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

# Final model, using the best depth from tuning
model = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=0)
model.fit(Xtrain, ytrain)
ypred = model.predict(Xtest)

print("FINAL MODEL | Features:", X.shape[1])
print("Accuracy:", round(metrics.accuracy_score(ytest, ypred), 3))
print(metrics.classification_report(ytest, ypred, target_names=["Fast", "Slow"]))

Features: 77
Accuracy: 0.66
              precision    recall  f1-score   support

        Fast       0.65      0.69      0.67     51448
        Slow       0.67      0.63      0.65     51427

    accuracy                           0.66    102875
   macro avg       0.66      0.66      0.66    102875
weighted avg       0.66      0.66      0.66    102875



# **Random Forest Algorithm Analysis**
- **Why we chose it:** We wanted to know what causes some 311 requests to be answered slowly: where the request is, what kind of request it is, or which department handles it. The Python Data Science Handbook (VanderPlas, Ch. 5, "In-Depth: Decision Trees and Random Forests") describes a random forest as an *ensemble* of decision trees, where each tree asks a series of yes/no questions and the trees vote, so "the sum can be greater than the parts." This fits our data well for three reasons from the textbook: (1) it is *nonparametric and flexible*, so it can pick up combinations like district + year without us defining them, which matters since most of our columns are categories; (2) training and prediction are *very fast and can run in parallel* because each tree is independent, which matters with over 500,000 requests; and (3) the trees' votes give a *probability* for each request (`predict_proba`), which we used to compare districts and years. Its feature importance also ranks which factors matter most, which directly answers our Q2. Random Forest is an outside-class algorithm (Breiman, 2001; VanderPlas, Ch. 5).

- **Tuning / training:** We used closed requests only (513,830), labeled each one slow if it took longer than the median for its request category, and split the data 80% training / 20% testing. Our baseline model (district, type, department, season, 2024 switch) reached 61% accuracy. The textbook warns that single decision trees overfit easily ("it is very easy to go too deep in the tree"), and that random forests reduce this by averaging many randomized trees (bagging). We tested this by tuning tree depth from 5 to unlimited: accuracy leveled off at about 61% after depth 15, and training and test accuracy stayed within 0.01 of each other at every depth, so the forest did not overfit, as the textbook predicts. Since tuning didn't help further, we added better features (the year and month each request was made), which raised accuracy to 66%. Final model: 100 trees, max depth 20, 77 features.

- **Metrics (effectiveness):** Accuracy 0.66 vs. 0.50 for random guessing. Precision 0.65 fast / 0.67 slow, recall 0.69 fast / 0.63 slow, F1 0.67 / 0.65. The model's predicted slow rate for each district came within about 1 point of the actual rate, showing its probability estimates are reliable. Overall, the model is moderately effective: it clearly beats guessing, trains in seconds on our full dataset, and doesn't overfit, but 66% accuracy means the factors in our data explain only part of why requests are slow. Other things we don't have data on, like crew availability or how complex each problem is, likely explain the rest.

- **What it tells us about our question:** Request type matters most (31%), followed by year (19%), department (15%), location (13%), and month (13%). Timing as a whole (year, month, season, 2024 switch) adds up to about 40%. So slow service in Buffalo is explained more by what people request and when than by where they live, which matches Hamel and Holliday's findings in other cities. Location still matters, though: North's requests were slow 54.5% of the time compared to similar requests, while every other district was between 48% and 51%. The heatmap shows service got slower in every district from 2020 to 2024, peaked in 2024 (the year Buffalo switched 311 systems), then improved in 2025 and 2026. **Limitation:** the textbook notes that random forests' main weakness is that "the results are not easily interpretable." We addressed this by grouping feature importance into categories and charting predicted probabilities by district and year, but importance can still favor categories with many options, so we treat the ranking as a strong signal rather than an exact measure.

# Algorithm 2: k-Means
**Question it answers:** Which neighborhoods share similar service patterns, and what do the slowest areas have in common? (Q4)

In [42]:
# k-Means: your code here

### k-Means Write-up
- **Why we chose it:**
- **Tuning / training:**
- **Metrics:**
- **What it tells us about our question:**